In [2]:
pip install pulp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 86.9 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.


In [103]:
import random
import math
from dataclasses import dataclass, field
from typing import List, Tuple

import pulp

# Define the Black-Scholes formulas for call and put options
def black_scholes_call(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.0
    d1 = (math.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    call_price = S * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)
    return call_price

def black_scholes_put(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.0
    d1 = (math.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    put_price = K * math.exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)
    return put_price

def norm_cdf(x):
    """
    Cumulative distribution function for the standard normal distribution.
    """
    return (1.0 + math.erf(x / math.sqrt(2.0))) / 2.0

@dataclass
class OptionData:
    """
    Represents the data for an option.
    """
    name: str
    s: float          # Underlying asset price (S)
    k: float          # Strike price (K)
    t: float          # Time to maturity in years (T)
    r: float          # Risk-free interest rate (r)
    sigma: float      # Volatility of the underlying asset (σ)
    option_type: str  # Option type: "call" or "put"
    market_price: float  # Current market price of the option

@dataclass
class Portfolio:
    """
    Manages the portfolio's holdings.
    """
    holdings: List[Tuple[str, float]] = field(default_factory=list)  # List of (option name, position size)

def compute_theoretical_prices(option_data: List[OptionData]) -> List[float]:
    """
    Computes theoretical option prices using the Black-Scholes model.
    """
    theoretical_prices = []
    for option in option_data:
        if option.option_type.lower() == "call":
            price = black_scholes_call(option.s, option.k, option.t, option.r, option.sigma)
        else:
            price = black_scholes_put(option.s, option.k, option.t, option.r, option.sigma)
        theoretical_prices.append(price)
    return theoretical_prices

def find_arbitrage(
    market_prices: List[float],
    transaction_costs: List[float],
    capital: float,
    liquidity: List[float],
    index_returns: List[float],
    risk_levels: List[float],
    option_data: List[OptionData],
) -> List[float]:
    """
    Finds arbitrage opportunities and computes optimal portfolio weights using linear programming.
    """
    num_assets = len(market_prices)
    num_states = len(index_returns)

    # Replace infinite liquidity with a large finite number
    LARGE_NUMBER = 1e12
    adjusted_liquidity = [min(LARGE_NUMBER, liq) for liq in liquidity]

    # Initialize the LP problem
    prob = pulp.LpProblem("ArbitrageOptimization", pulp.LpMaximize)

    # Decision variables
    weights = [
        pulp.LpVariable(
            f"w_{i}",
            lowBound=-adjusted_liquidity[i],
            upBound=adjusted_liquidity[i],
            cat='Continuous'
        ) for i in range(num_assets)
    ]
    w_plus = [
        pulp.LpVariable(
            f"w_plus_{i}",
            lowBound=0,
            upBound=adjusted_liquidity[i],
            cat='Continuous'
        ) for i in range(num_assets)
    ]
    w_minus = [
        pulp.LpVariable(
            f"w_minus_{i}",
            lowBound=0,
            upBound=adjusted_liquidity[i],
            cat='Continuous'
        ) for i in range(num_assets)
    ]

    # Equality constraints: w_i = w_plus_i - w_minus_i
    for i in range(num_assets):
        prob += (weights[i] == w_plus[i] - w_minus[i]), f"NetPositionConstraint_{i}"

    # Compute theoretical prices using the Black-Scholes model
    theoretical_prices = compute_theoretical_prices(option_data)

    # Build the objective function (profit maximization)
    profit_per_unit = [
        theoretical_prices[i] - market_prices[i] - transaction_costs[i]
        for i in range(num_assets)
    ]
    prob += pulp.lpSum([profit_per_unit[i] * weights[i] for i in range(num_assets)]), "TotalExpectedProfit"

    # Capital constraint: limit total investment to capital
    total_investment = pulp.lpSum([
        (w_plus[i] + w_minus[i]) * (market_prices[i] + transaction_costs[i])
        for i in range(num_assets)
    ])
    prob += (total_investment <= capital), "CapitalConstraint"

    # Liquidity constraints: w_plus_i ≤ adjusted_liquidity_i, w_minus_i ≤ adjusted_liquidity_i
    # These constraints are redundant because we set the variable bounds accordingly, but we can include them for clarity
    for i in range(num_assets):
        prob += (w_plus[i] <= adjusted_liquidity[i]), f"LiquidityConstraintPlus_{i}"
        prob += (w_minus[i] <= adjusted_liquidity[i]), f"LiquidityConstraintMinus_{i}"

    # Stochastic dominance constraints
    # In this simplified example, we assume portfolio returns are the same across states
    portfolio_return = pulp.lpSum([
        weights[i] * (theoretical_prices[i] - market_prices[i] - transaction_costs[i])
        for i in range(num_assets)
    ])

    for risk_level in risk_levels:
        for s in range(num_states):
            index_return = index_returns[s]
            prob += (
                portfolio_return * risk_level >= index_return * risk_level,
                f"StochasticDominance_{risk_level}_{s}"
            )

    # Position limit constraints
    num_options = num_assets
    max_investment_per_option = capital / num_options
    for i in range(num_assets):
        investment_in_option = weights[i] * (market_prices[i] + transaction_costs[i])
        prob += (
            investment_in_option <= max_investment_per_option,
            f"PositionLimitUpper_{i}"
        )
        prob += (
            investment_in_option >= -max_investment_per_option,
            f"PositionLimitLower_{i}"
        )

    # Solve the optimization problem
    prob.solve()

    # Check the solution status
    if pulp.LpStatus[prob.status] != 'Optimal':
        raise Exception("No optimal solution found.")

    # Retrieve final positions (weights) for each option
    optimal_weights = [weights[i].varValue for i in range(num_assets)]
    return optimal_weights

def construct_portfolio(
    option_data: List[OptionData],
    capital: float,
    risk_levels: List[float],
    index_returns: List[float],
    transaction_costs: List[float],
    liquidity: List[float],
) -> Portfolio:
    """
    Constructs the portfolio by finding optimal weights and assembling holdings.
    """
    market_prices = [option.market_price for option in option_data]

    # Find optimal portfolio weights via linear programming
    portfolio_weights = find_arbitrage(
        market_prices,
        transaction_costs,
        capital,
        liquidity,
        index_returns,
        risk_levels,
        option_data,
    )

    # Create portfolio holdings
    holdings = [
        (option.name, weight)
        for option, weight in zip(option_data, portfolio_weights)
    ]

    return Portfolio(holdings=holdings)

In [104]:
def main():
    # Read the JSON file containing the options data
    file_path = "options_data.json"  # Replace with the correct path to your JSON file
    with open(file_path, "r") as file:
        data = json.load(file)

    MIN_SIGMA = 1e-5  # Minimum volatility to avoid division by zero
    MIN_T = 1e-5      # Minimum time to avoid division by zero

    # Convert JSON data to a list of OptionData objects
    option_data_list: List[OptionData] = []
    for option_json in data:
        name = option_json.get("symbol")
        s = float(option_json.get("underlyingPrice", 0))
        k = extract_strike(name)  # Extract strike price from the symbol
        if k == 0.0:
            continue  # Skip options with invalid strike price

        # Assuming a default of 30 days to expiration if missing
        days_to_expiration = option_json.get("daysToExpiration", 30)
        t = float(days_to_expiration) / 365  # Convert days to years
        t = max(t, MIN_T)  # Ensure t is at least MIN_T

        r = 0.05  # Assuming a risk-free interest rate of 5%

        # Handle missing or zero implied volatility
        mark_iv = option_json.get("markIv")
        if mark_iv is None or float(mark_iv) == 0:
            sigma = 0.2  # Assign a default volatility of 20%
        else:
            sigma = float(mark_iv)
            if sigma > 1:  # If sigma is in percentage, convert it to decimal
                sigma /= 100
        sigma = max(sigma, MIN_SIGMA)  # Ensure sigma is at least MIN_SIGMA

        option_type = "call" if "-C" in name else "put"
        market_price = float(option_json.get("markPrice", 0))
        if market_price <= 0:
            # If market price is zero, estimate it using Black-Scholes formula
            if option_type == 'call':
                market_price = black_scholes_call(s, k, t, r, sigma)
            else:
                market_price = black_scholes_put(s, k, t, r, sigma)

        option_data = OptionData(
            name=name,
            s=s,
            k=k,
            t=t,
            r=r,
            sigma=sigma,
            option_type=option_type,
            market_price=market_price
        )
        option_data_list.append(option_data)

    for option_data in option_data_list[:5]:  # Print first 5 options
        print(f"Option: {option_data.name}, Type: {option_data.option_type}, Strike: {option_data.k}, Market Price: {option_data.market_price}, Sigma: {option_data.sigma}, T: {option_data.t}")

    # Split options into calls and puts
    calls = [option for option in option_data_list if option.option_type == "call"]
    puts = [option for option in option_data_list if option.option_type == "put"]

    # Total capital for the portfolio
    capital = 100_000.0

    # Risk levels for various investor profiles
    risk_levels = [0.95, 0.9, 0.8]

    # Read index price data from 'index_price.json' and compute index returns
    with open('index_price.json', 'r') as file:
        index_data = json.load(file)

    # Extract closing prices
    index_prices = index_data[0]['series']['prices']

    # Compute index returns
    index_returns = []
    for i in range(1, len(index_prices)):
        index_return = (index_prices[i] / index_prices[i - 1]) - 1
        index_returns.append(index_return)

    # Ensure that index_returns has the same number of states as needed
    num_states = max(len(index_returns), 50)  # Ensure at least 50 states
    if len(index_returns) < num_states:
        index_returns.extend(index_returns[:num_states - len(index_returns)])
    else:
        index_returns = index_returns[:num_states]

    # Transaction costs per option
    transaction_costs = [0.05] * len(option_data_list)

    # Liquidity constraints (using a large finite number instead of infinity)
    liquidity = [1e12] * len(option_data_list)

    # Construct the portfolio
    portfolio = construct_portfolio(
        option_data_list,
        capital,
        risk_levels,
        index_returns,
        transaction_costs,
        liquidity,
    )

    # Sort the portfolio holdings by strike price (extracted from the option name)
    portfolio.holdings.sort(key=lambda x: extract_strike(x[0]))  # Sort by strike price

    # Output the portfolio holdings for calls and puts separately
    print("\nOptimal Portfolio Holdings - Calls (sorted by strike price):")
    for name, position in portfolio.holdings:
        if "-C" in name:
            print(f"Option: {name}, Position: {position:.2f}")

    print("\nOptimal Portfolio Holdings - Puts (sorted by strike price):")
    for name, position in portfolio.holdings:
        if "-P" in name:
            print(f"Option: {name}, Position: {position:.2f}")

if __name__ == "__main__":
    main()


Option: BTC-11OCT24-62000-P, Type: put, Strike: 62000.0, Market Price: 1182.10143734, Sigma: 0.4257, T: 0.0821917808219178
Option: BTC-11OCT24-59000-C, Type: call, Strike: 59000.0, Market Price: 3622.62643226, Sigma: 0.473, T: 0.0821917808219178
Option: BTC-11OCT24-61000-C, Type: call, Strike: 61000.0, Market Price: 2070.75534707, Sigma: 0.4367, T: 0.0821917808219178
Option: BTC-11OCT24-50000-P, Type: put, Strike: 50000.0, Market Price: 12.75945005, Sigma: 0.7266, T: 0.0821917808219178
Option: BTC-11OCT24-48000-C, Type: call, Strike: 48000.0, Market Price: 14281.38840226, Sigma: 0.7903, T: 0.0821917808219178
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/huenique/anaconda3/envs/personal/lib/python3.12/site-packages/pulp/solverdir/cbc/linux/64/cbc /tmp/a28762569f1747839451836acb4b5cee-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /tmp/a28762569f1747839451836acb4b5cee-pulp.sol (default strategy 1)
At line 2 NAME   

In [109]:
import math
from dataclasses import dataclass, field
from typing import List, Tuple

import pulp
from scipy.stats import norm
from scipy.optimize import brentq

# Define the Black-Scholes formulas for call and put options
def black_scholes_call(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.0
    d1 = (math.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    call_price = S * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)
    return call_price

def black_scholes_put(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.0
    d1 = (math.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    put_price = K * math.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return put_price

# Function to compute implied volatility
def implied_volatility(option_type, market_price, S, K, T, r):
    def objective_function(sigma):
        if sigma <= 0:
            return market_price - 0.0
        if option_type.lower() == "call":
            return black_scholes_call(S, K, T, r, sigma) - market_price
        else:
            return black_scholes_put(S, K, T, r, sigma) - market_price

    try:
        # Implied volatility must be between 1e-5 and 5.0 (500%)
        implied_vol = brentq(objective_function, 1e-5, 5.0)
    except ValueError:
        implied_vol = 0.0  # If no solution found, set to zero
    return implied_vol

@dataclass
class OptionData:
    """
    Represents the data for an option.
    """
    name: str
    s: float          # Underlying asset price (S)
    k: float          # Strike price (K)
    t: float          # Time to maturity in years (T)
    r: float          # Risk-free interest rate (r)
    sigma: float      # Volatility of the underlying asset (σ)
    option_type: str  # Option type: "call" or "put"
    market_price: float  # Current market price of the option

@dataclass
class Portfolio:
    """
    Manages the portfolio's holdings.
    """
    holdings: List[Tuple[str, float]] = field(default_factory=list)  # List of (option name, position size)

def compute_probabilities_itm(option_data: List[OptionData]) -> List[float]:
    probabilities = []
    for option in option_data:
        d2 = (math.log(option.s / option.k) + (option.r - 0.5 * option.sigma ** 2) * option.t) / (option.sigma * math.sqrt(option.t))
        if option.option_type.lower() == "call":
            prob_itm = norm.cdf(d2)
        else:
            prob_itm = norm.cdf(-d2)
        probabilities.append(prob_itm)
    return probabilities

def find_optimal_portfolio(
    market_prices: List[float],
    transaction_costs: List[float],
    capital: float,
    liquidity: List[float],
    option_data: List[OptionData],
) -> List[float]:
    """
    Computes optimal portfolio weights using linear programming.
    """
    num_assets = len(market_prices)

    # Initialize the LP problem
    prob = pulp.LpProblem("OptionPortfolioOptimization", pulp.LpMaximize)

    # Decision variables (long-only positions)
    weights = [
        pulp.LpVariable(
            f"w_{i}",
            lowBound=0,
            upBound=liquidity[i],
            cat='Continuous'
        ) for i in range(num_assets)
    ]

    # Compute theoretical prices as expected payoffs
    theoretical_prices = compute_theoretical_prices(option_data)
    expected_payoffs = theoretical_prices

    # Build the objective function (expected profit maximization)
    expected_profits = [
        expected_payoffs[i] - market_prices[i] - transaction_costs[i]
        for i in range(num_assets)
    ]
    prob += pulp.lpSum([expected_profits[i] * weights[i] for i in range(num_assets)]), "ExpectedProfit"

    # Capital constraint: limit total investment to capital
    total_investment = pulp.lpSum([
        (market_prices[i] + transaction_costs[i]) * weights[i]
        for i in range(num_assets)
    ])
    prob += (total_investment <= capital), "CapitalConstraint"

    # Position limit constraints
    max_option_allocation = 0.05 * capital  # Adjust as appropriate
    for i in range(num_assets):
        investment_in_option = (market_prices[i] + transaction_costs[i]) * weights[i]
        prob += (
            investment_in_option <= max_option_allocation,
            f"MaxOptionAllocation_{i}"
        )

    # Risk constraint: limit total weighted volatility
    total_weighted_volatility = pulp.lpSum([
        weights[i] * option_data[i].sigma
        for i in range(num_assets)
    ])
    max_total_volatility = 0.2 * capital  # Adjust as appropriate
    prob += total_weighted_volatility <= max_total_volatility, "MaxTotalVolatility"

    # Solve the optimization problem
    prob.solve()

    # Check the solution status
    if pulp.LpStatus[prob.status] != 'Optimal':
        print("No optimal solution found.")
        return [0.0] * num_assets

    # Retrieve final positions (weights) for each option
    optimal_weights = [weights[i].varValue for i in range(num_assets)]
    return optimal_weights

def construct_portfolio(
    option_data: List[OptionData],
    capital: float,
    transaction_costs: List[float],
    liquidity: List[float],
) -> Portfolio:
    """
    Constructs the portfolio by finding optimal weights and assembling holdings.
    """
    market_prices = [option.market_price for option in option_data]

    # Find optimal portfolio weights via linear programming
    portfolio_weights = find_optimal_portfolio(
        market_prices,
        transaction_costs,
        capital,
        liquidity,
        option_data,
    )

    # Create portfolio holdings
    holdings = [
        (option.name, weight)
        for option, weight in zip(option_data, portfolio_weights)
        if weight is not None and weight > 0
    ]

    return Portfolio(holdings=holdings)

def main():
    import json
    # Read the JSON file containing the options data
    file_path = "options_data.json"  # Replace with the correct path to your JSON file
    with open(file_path, "r") as file:
        data = json.load(file)

    MIN_SIGMA = 1e-5  # Minimum volatility to avoid division by zero
    MAX_SIGMA = 5.0   # Maximum volatility to avoid unrealistic values
    MIN_T = 1e-5      # Minimum time to avoid division by zero

    # Function to extract strike price from the option symbol
    def extract_strike(name):
        # Assumes the symbol format is 'BTC-25OCT24-82000-P'
        try:
            parts = name.split('-')
            strike_price = float(parts[2])
        except (IndexError, ValueError):
            strike_price = 0.0
        return strike_price

    # Convert JSON data to a list of OptionData objects
    option_data_list: List[OptionData] = []
    for option_json in data:
        name = option_json.get("symbol")
        s = float(option_json.get("underlyingPrice", 0))
        k = extract_strike(name)
        if k == 0.0:
            continue  # Skip options with invalid strike price

        # Assuming a default of 30 days to expiration if missing
        days_to_expiration = option_json.get("daysToExpiration", 30)
        t = float(days_to_expiration) / 365  # Convert days to years
        t = max(t, MIN_T)  # Ensure t is at least MIN_T

        r = 0.05  # Assuming a risk-free interest rate of 5%

        # Get market price
        market_price = float(option_json.get("markPrice", 0))
        if market_price <= 0:
            continue  # Skip options with zero market price

        option_type = "call" if "-C" in name else "put"

        # Compute implied volatility to match market price
        sigma = implied_volatility(option_type, market_price, s, k, t, r)
        sigma = max(min(sigma, MAX_SIGMA), MIN_SIGMA)  # Ensure sigma is within bounds
        if sigma <= MIN_SIGMA or sigma >= MAX_SIGMA:
            continue  # Skip options with unrealistic implied volatility

        option_data = OptionData(
            name=name,
            s=s,
            k=k,
            t=t,
            r=r,
            sigma=sigma,
            option_type=option_type,
            market_price=market_price
        )
        option_data_list.append(option_data)

    # Print the number of options parsed
    print(f"Number of options parsed: {len(option_data_list)}")

    for option_data in option_data_list[:5]:  # Print first 5 options
        print(f"Option: {option_data.name}, Type: {option_data.option_type}, Strike: {option_data.k}, Market Price: {option_data.market_price}, Sigma: {option_data.sigma}, T: {option_data.t}")

    # Transaction costs per option
    transaction_costs = [0.05] * len(option_data_list)

    # Liquidity constraints (using a large finite number instead of infinity)
    liquidity = [1e12] * len(option_data_list)

    # Total capital for the portfolio
    capital = 100_000.0

    # Construct the portfolio
    portfolio = construct_portfolio(
        option_data_list,
        capital,
        transaction_costs,
        liquidity,
    )

    # Function to extract strike price for sorting
    def extract_strike_for_sorting(name):
        # Assumes the symbol format is 'BTC-25OCT24-82000-P'
        try:
            parts = name.split('-')
            strike_price = float(parts[2])
            return strike_price
        except (IndexError, ValueError):
            return 0.0

    # Sort the portfolio holdings by strike price
    portfolio.holdings.sort(key=lambda x: extract_strike_for_sorting(x[0]))  # Sort by strike price

    # Output the portfolio holdings
    print("\nOptimal Portfolio Holdings:")
    for name, position in portfolio.holdings:
        print(f"Option: {name}, Position: {position:.2f}")

if __name__ == "__main__":
    main()


Number of options parsed: 49
Option: BTC-11OCT24-62000-P, Type: put, Strike: 62000.0, Market Price: 1182.10143734, Sigma: 0.20179474299496727, T: 0.0821917808219178
Option: BTC-11OCT24-59000-C, Type: call, Strike: 59000.0, Market Price: 3622.62643226, Sigma: 0.1500469632518859, T: 0.0821917808219178
Option: BTC-11OCT24-61000-C, Type: call, Strike: 61000.0, Market Price: 2070.75534707, Sigma: 0.1636779669290725, T: 0.0821917808219178
Option: BTC-11OCT24-50000-P, Type: put, Strike: 50000.0, Market Price: 12.75945005, Sigma: 0.3211318729968601, T: 0.0821917808219178
Option: BTC-11OCT24-80000-C, Type: call, Strike: 80000.0, Market Price: 2.55309366, Sigma: 0.29008159793876054, T: 0.0821917808219178
Option: BTC-11OCT24-62000-P, Theoretical Price: 1182.10, Market Price: 1182.10, Sigma: 0.2018
Option: BTC-11OCT24-59000-C, Theoretical Price: 3622.63, Market Price: 3622.63, Sigma: 0.1500
Option: BTC-11OCT24-61000-C, Theoretical Price: 2070.76, Market Price: 2070.76, Sigma: 0.1637
Option: BTC-11